In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from datetime import datetime

from sklearn.model_selection import train_test_split, GroupKFold, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone

import optuna
import xgboost as xgb
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# ========================= CONFIG =========================
DATA_PATH = r"A:\College\Thesis\SHD\USstock\baseline_metadata_and_annual_results_hdd65.parquet"
MODEL_SAVE_DIR = Path(r"A:\College\Thesis\Submission\models\resstock_5.8")
PLOTS_DIR = MODEL_SAVE_DIR / "plots"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CV_FOLDS = 5
MIN_SAMPLES_PER_ZONE = 250
N_TRIALS_XGB = 30
N_TRIALS_CAT_GLOBAL = 15
N_TRIALS_CAT_ZONE   = 30
N_TRIALS_LGB = 30
USE_LOG_TARGET = True
FT2_PER_M2 = 10.7639104167
EARLY_STOPPING_ROUNDS = 50
MAX_ESTIMATORS = 2000


# ---- Envelope physics constants ----------------
_CEIL_HT_FT          = 9.0
_CEIL_HT_M           = _CEIL_HT_FT * 0.3048
_R_FILM              = 0.68 + 0.17          # interior + exterior air films [hr·ft²·°F/BTU]
_BASE_R_WALL         = 3.0                  # Baseline R-value for framing, sheathing, siding
_BASE_R_CEIL         = 2.0                  # Baseline R-value for ceiling drywall
_BASE_R_ROOF         = 2.0                  # Baseline R-value for roof sheathing/shingles
_BASE_R_FLOOR        = 2.0                  # Baseline R-value for floor structures
_ACH50_DIV           = 25                   # natural infiltration ≈ ACH50 / 20
_BTU_TO_KWH          = 2.931e-4
_KBTU_TO_KWH         = 1000 * _BTU_TO_KWH   # 0.2931 kWh / kBtu
_BTU_PER_HR_F_TO_W_K = 0.52752
_RHO_CP              = 0.33e-3               # ρ·cp_air [kWh/(m³·K)]
_ETA_IG              = 0.85                  # internal-gain utilisation factor (ISO 13790)
_MECH_FRAC   = 0           # fraction of natural ACH attributed to mechanical ventilation


# _W_PER_OCC = 80   #(ASHRAE 55)
# _OCC_EFLH  = 4380 #hr/yr (half-time at home)

# ---- Window U-value lookup [BTU/hr·ft²·°F] --------------------------------
_U_WIN_MAP = {
    "single, clear, metal":                                1.10,
    "single, clear, metal, exterior clear storm":          0.82,
    "single, clear, non-metal":                            0.98,
    "single, clear, non-metal, exterior clear storm":      0.74,
    "double, clear, metal, air":                           0.65,
    "double, clear, metal, air, exterior clear storm":     0.52,
    "double, clear, non-metal, air":                       0.49,
    "double, clear, non-metal, air, exterior clear storm": 0.40,
    "double, low-e, non-metal, air, m-gain":               0.32,
    "triple, low-e, non-metal, air, l-gain":               0.22,
}
_U_WIN_DEFAULT = 0.49
# ===========================================================================

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# ========================= HELPERS =========================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def make_actual_vs_pred_plot(y_true, y_pred, title, out_path):
    plt.figure(figsize=(7, 7))
    plt.scatter(y_true, y_pred, s=8, alpha=0.35)
    mn = min(np.min(y_true), np.min(y_pred))
    mx = max(np.max(y_true), np.max(y_pred))
    plt.plot([mn, mx], [mn, mx], "r--", linewidth=2)
    plt.xlabel("Actual specific heating demand (kWh/m²/yr)")
    plt.ylabel("Predicted specific heating demand (kWh/m²/yr)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close()

def get_cv(groups, n_splits=5, random_state=42):
    if groups is not None:
        return GroupKFold(n_splits=n_splits)
    return KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

def randomize_groups(groups, random_state=42):
    rng = np.random.RandomState(random_state)
    uniq = np.unique(groups)
    shuffled = rng.permutation(uniq)
    mapping = {g: i for i, g in enumerate(shuffled)}
    return np.vectorize(mapping.get)(groups)

def build_preprocess_ohe(numeric_features, categorical_features):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

def fit_transform_catboost(X, num_feats, cat_feats):
    num_imp = SimpleImputer(strategy="median")
    cat_imp = SimpleImputer(strategy="constant", fill_value="__MISSING__")
    X_num = pd.DataFrame(num_imp.fit_transform(X[num_feats]), columns=num_feats, index=X.index)
    X_cat = pd.DataFrame(cat_imp.fit_transform(X[cat_feats]), columns=cat_feats, index=X.index).astype("string")
    X_out = pd.concat([X_num, X_cat], axis=1)
    cat_feature_indices = [X_out.columns.get_loc(c) for c in cat_feats]
    return X_out, num_imp, cat_imp, cat_feature_indices

def transform_catboost(X, num_feats, cat_feats, num_imp, cat_imp):
    X_num = pd.DataFrame(num_imp.transform(X[num_feats]), columns=num_feats, index=X.index)
    X_cat = pd.DataFrame(cat_imp.transform(X[cat_feats]), columns=cat_feats, index=X.index).astype("string")
    X_out = pd.concat([X_num, X_cat], axis=1)
    return X_out
 # Parsers
 
def parse_r_value(s):
    if pd.isna(s):
        return np.nan
    txt = str(s).strip().lower()
    if txt in ("none", "uninsulated", "__missing__", ""):
        return 0.0
    m = re.search(r'r-?(\d+\.?\d*)', txt, re.IGNORECASE)
    return float(m.group(1)) if m else 0.0

def parse_ach50(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+\.?\d*)\s*ach50', str(s), re.IGNORECASE)
    return float(m.group(1)) if m else np.nan

def parse_duct_leakage(s):
    if pd.isna(s):
        return np.nan
    txt = str(s).strip().lower()
    if txt.startswith("none") or txt == "0% leakage to outside, uninsulated":
        return 0.0
    m = re.search(r'(\d+)%', txt)
    return float(m.group(1)) / 100.0 if m else np.nan

# def parse_occupants(s):
#     if pd.isna(s):
#         return np.nan
#     txt = str(s).strip()
#     if txt == "10+":
#         return 10.0
#     try:
#         return float(txt)
#     except ValueError:
#         return np.nan

def decode_vintage(s):
    VINTAGE_MAP = {
        "<1940": 1930, "1940s": 1945, "1950s": 1955, "1960s": 1965,
        "1970s": 1975, "1980s": 1985, "1990s": 1995, "2000s": 2005, "2010s": 2015,
    }
    if pd.isna(s):
        return np.nan
    return VINTAGE_MAP.get(str(s).strip(), np.nan)

_LPD_MAP = {
    "100% led":          4.0,
    "100% cfl":          7.5,
    "100% incandescent": 14.0,
}

print("=== Loading parquet ===")
df = pd.read_parquet(DATA_PATH).convert_dtypes()
if df.index.name == "bldg_id" or "bldg_id" not in df.columns:
    df = df.reset_index()
print(f"Raw shape: {df.shape}")

df["heating_energy_kwh"] = safe_numeric(
    df["out.load.heating.energy_delivered.kbtu"]
) * _KBTU_TO_KWH

# No fall back to summing fuel types here like comstock, the dataset is saturated enough. Drop out.load nans
df["floor_area_ft2"] = safe_numeric(df["in.sqft"])
df = df.dropna(subset=["heating_energy_kwh", "floor_area_ft2"]).copy()
df = df[(df["floor_area_ft2"] > 0) & (df["heating_energy_kwh"] >= 0)].copy()
df["floor_area_m2"] = df["floor_area_ft2"] / FT2_PER_M2
df["specific_heat_demand_kwh_m2"] = df["heating_energy_kwh"] / df["floor_area_m2"]
df = df[np.isfinite(df["specific_heat_demand_kwh_m2"])].copy()

if "upgrade" in df.columns:
    u = df["upgrade"].astype("string").str.strip().str.lower()
    is_baseline = u.isin(["0", "00", "baseline", "base", "upgrade00"]) | u.str.fullmatch(r"0+")
    if is_baseline.sum() > 1000:
        df = df[is_baseline].copy()

CLIMATE_COL = "in.building_america_climate_zone"


if "hdd65_annual_avg" in df.columns:
    df["hdd65"] = safe_numeric(df["hdd65_annual_avg"])
else:
    raise KeyError("'hdd65_annual_avg' column not found. Check parquet schema.")

if "in.vintage" in df.columns:
    df["year_built_numeric"] = df["in.vintage"].apply(decode_vintage)

if "in.infiltration" in df.columns:
    df["ach50_numeric"] = df["in.infiltration"].apply(parse_ach50)
    df["ach_natural"]   = df["ach50_numeric"] / _ACH50_DIV

for col, out in [
    ("in.insulation_wall",            "r_wall"),
    ("in.insulation_ceiling",         "r_ceiling"),
    ("in.insulation_roof",            "r_roof"),
    ("in.insulation_floor",           "r_floor"),
    ("in.insulation_foundation_wall", "r_foundation_wall"),
    ("in.insulation_slab",            "r_slab"),
    ("in.insulation_rim_joist",       "r_rim_joist"), 
]:
    if col in df.columns:
        df[out] = df[col].apply(parse_r_value)

if "in.duct_leakage_and_insulation" in df.columns:
    df["duct_leakage_frac"] = df["in.duct_leakage_and_insulation"].apply(parse_duct_leakage)

# if "in.occupants" in df.columns:
#     df["occupants_numeric"] = df["in.occupants"].apply(parse_occupants)

# ==================== ENVELOPE AREAS ====================

if "out.params.wall_area_above_grade_conditioned_ft_2" in df.columns:
    df["wall_area_ft2"] = safe_numeric(df["out.params.wall_area_above_grade_conditioned_ft_2"])
else:
    df["wall_area_ft2"] = 4.0 * np.sqrt(df["floor_area_ft2"]) * _CEIL_HT_FT

if "out.params.window_area_ft_2" in df.columns:
    df["win_area_ft2"] = safe_numeric(df["out.params.window_area_ft_2"])
else:
    _WWR_FALLBACK = 0.15
    df["win_area_ft2"] = _WWR_FALLBACK * df["wall_area_ft2"]

df["net_wall_area_ft2"] = (df["wall_area_ft2"] - df["win_area_ft2"]).clip(lower=0)

if "out.params.roof_area_ft_2" in df.columns:
    df["ceil_area_ft2"] = safe_numeric(df["out.params.roof_area_ft_2"])
else:
    df["ceil_area_ft2"] = df["floor_area_ft2"]

if "in.windows" in df.columns:
    _win_norm = df["in.windows"].astype("string").str.lower().str.strip()
    df["u_win_btu"] = _win_norm.map(_U_WIN_MAP)
else:
    df["u_win_btu"] = np.nan

df["wwr_actual"] = np.where(
    df["wall_area_ft2"] > 0,
    df["win_area_ft2"] / df["wall_area_ft2"],
    np.nan,
)

# ---- U-values from R-values ----
if "r_wall" in df.columns:
    df["u_wall"] = 1.0 / (df["r_wall"] + _BASE_R_WALL + _R_FILM)
if "r_ceiling" in df.columns:
    df["u_ceiling"] = 1.0 / (df["r_ceiling"] + _BASE_R_CEIL + _R_FILM)
if "r_roof" in df.columns:
    df["u_roof"] = 1.0 / (df["r_roof"] + _BASE_R_ROOF + _R_FILM)
if "r_floor" in df.columns:
    df["u_floor"] = 1.0 / (df["r_floor"] + _BASE_R_FLOOR + _R_FILM)

# ---- UA products ----
ua_terms = []

if "u_wall" in df.columns:
    df["UA_wall"] = df["u_wall"] * df["net_wall_area_ft2"]
    ua_terms.append("UA_wall")

# Row-wise fallback for roof vs ceiling thermal boundary
if "u_ceiling" in df.columns and "u_roof" in df.columns:
    _ceil_u = df[["u_ceiling", "u_roof"]].min(axis=1)
elif "u_ceiling" in df.columns:
    _ceil_u = df["u_ceiling"]
elif "u_roof" in df.columns:
    _ceil_u = df["u_roof"]
else:
    _ceil_u = None

if _ceil_u is not None:
    df["UA_ceil"] = _ceil_u * df["ceil_area_ft2"]
    ua_terms.append("UA_ceil")

if "u_floor" in df.columns:
    df["UA_floor"] = df["u_floor"] * df["floor_area_ft2"]
    ua_terms.append("UA_floor")

df["UA_win"] = df["u_win_btu"] * df["win_area_ft2"]
ua_terms.append("UA_win")

if ua_terms:
    df["UA_total"] = df[ua_terms].sum(axis=1, min_count=len(ua_terms))
    df["UA_per_area"] = (df["UA_total"] * _BTU_PER_HR_F_TO_W_K) / df["floor_area_m2"]

if "UA_per_area" in df.columns and "hdd65" in df.columns:
    df["HDD65_x_UA_per_area"] = (df["hdd65"] / 1.8) * df["UA_per_area"]


#   Q_heat = Q_trans + Q_vent + Q_inf - η · Q_int
if "building_fraction_heated" not in df.columns:
    df["building_fraction_heated"] = 1.0
else:
    df["building_fraction_heated"] = safe_numeric(df["building_fraction_heated"]).fillna(1.0)

# ── 1. Transmission losses ────────────────────────────────────────────────────
# Identical formula to ComStock:
#   UA_heated [BTU/hr·°F] × HDD65 [°F·day] × 24 h/day × BTU_TO_KWH / floor_area_m²
# UA_total is already computed above from wall/ceiling/window/floor components.

if "UA_total" in df.columns and "hdd65" in df.columns:
    _UA_heated = df["UA_total"] * df["building_fraction_heated"]
    _q_trans = (
        _UA_heated * df["hdd65"] * 24 * _BTU_TO_KWH
        / df["floor_area_m2"]
    )
else:
    _q_trans = pd.Series(np.nan, index=df.index)

# ── 2. Ventilation losses ─────────────────────────────────────────────────────
# ComStock drives this with design_outdoor_air_flow_rate [m³/m²/s] × f_occupancy × HDD_Khr.
# ResStock has no dedicated ventilation column, so we split ach_natural into:
#   - mechanical ventilation fraction → _q_vent  (always running, like ComStock's ODA)
#   - pure infiltration fraction       → _q_inf   (weather-driven, see §3)
#
# HDD_Khr: convert HDD65 [°F·day] → [K·hr/yr]  (÷1.8 × 24), same as ComStock.
# Volume per unit floor area uses the constant ceiling height.

if "ach_natural" in df.columns and "hdd65" in df.columns:
    _HDD_Khr = df["hdd65"] / 1.8 * 24        # [K·hr/yr] — identical to ComStock
    _vol_per_m2 = _CEIL_HT_M                  # m³ of air per m² of floor

    # Mechanical ventilation share (constant, schedule-independent for residential)
    _ach_mech = df["ach_natural"] * _MECH_FRAC
    _q_vent   = _RHO_CP * _ach_mech * _vol_per_m2 * _HDD_Khr
else:
    _q_vent = pd.Series(0.0, index=df.index)

# ── 3. Infiltration losses ────────────────────────────────────────────────────
# Pure weather-driven infiltration: the remaining (1 - _MECH_FRAC) share of ach_natural.
# Same RHO_CP × ACH × volume × HDD_Khr structure as ComStock's infiltration term.

if "ach_natural" in df.columns and "hdd65" in df.columns:
    _HDD_Khr = df["hdd65"] / 1.8 * 24        # may already exist; redefinition is harmless
    _ach_inf  = df["ach_natural"] * (1.0 - _MECH_FRAC)
    _q_inf    = _RHO_CP * _ach_inf * _vol_per_m2 * _HDD_Khr
else:
    _q_inf = pd.Series(0.0, index=df.index)

# ── 4. Internal gains ─────────────────────────────────────────────────────────
# ComStock uses: lighting LPD × EFLH + equipment EPD × EFLH + occupant density × W/occ × EFLH
# ResStock only provides a categorical lighting field (in.lighting) → mapped to W/ft²,
# then converted to W/m².  Equipment and occupant density are unavailable; those terms = 0.
#
# Annual equivalent full-load hours for residential lighting: ASHRAE 90.2 / RESNET default.
_LIGHTING_EFLH = 1200.0        # hr/yr — conservative residential assumption
_LPD_MAP = {                   # W/ft²  (same mapping already used for features above)
    "100% led":           4.0,
    "100% cfl":           7.5,
    "100% incandescent": 14.0,
}

_q_int = pd.Series(0.0, index=df.index)

if "in.lighting" in df.columns:
    _lpd_wpft2 = (
        df["in.lighting"]
        .astype("string").str.lower().str.strip()
        .map(_LPD_MAP)
    )
    # W/ft² → W/m²  (× FT2_PER_M2), then × EFLH × 1e-3 → kWh/m²/yr
    # Matches ComStock's LPD term exactly, just with a fixed EFLH instead of a simulated one.
    _q_int += (_lpd_wpft2 * FT2_PER_M2 * _LIGHTING_EFLH * 1e-3).fillna(0.0)

# Equipment (plug loads) — no ResStock column available; add here if data is extended:
# _EPD  = "in.misc_electric_loads_..w_per_ft2"   # hypothetical future column
# _EEFL = 4380  # hr/yr
# if _EPD in df.columns:
#     _q_int += (df[_EPD] * FT2_PER_M2 * _EEFL * 1e-3).fillna(0)



# if "occupants_numeric" in df.columns:
#     _occ_density = df["occupants_numeric"] / df["floor_area_m2"]  # ppl/m²
#     _q_int += (_occ_density * _W_PER_OCC * _OCC_EFLH * 1e-3).fillna(0)

#   SHD = (Q_trans + Q_vent + Q_inf - η · Q_int).clip(lower=0)

df["calculated_shd"] = np.where(
    _q_trans.notna(),
    (
        _q_trans.fillna(0)
        + _q_vent.fillna(0)
        + _q_inf.fillna(0)
        - _ETA_IG * _q_int.fillna(0)
    ).clip(lower=0),
    np.nan,
)
# ==================== NaN-DROP FOR CALCULATED FEATURES ====================
_CALC_FEATURES = [
    "calculated_shd",
    "UA_total",
    "UA_per_area",
    "HDD65_x_UA_per_area",
    "UA_wall",
    "UA_ceil",
    "UA_win",
]
_calc_drop_cols = [c for c in _CALC_FEATURES if c in df.columns]
_before = len(df)
df = df.dropna(subset=_calc_drop_cols).copy()
print(
    f"Dropped {_before - len(df):,} rows with NaN in calculated features "
    f"({len(df):,} remaining)"
)

# ==================== FEATURE LISTS ====================
numeric_features = [
    "floor_area_m2",
    "hdd65",
    "HDD65_x_UA_per_area",
    "UA_total",
    "UA_per_area",
    "r_wall",
    "r_ceiling",
    "r_roof",
    "r_floor",
    "r_foundation_wall",
    "r_slab",
    "r_rim_joist",
    "wall_area_ft2",
    "win_area_ft2",
    "net_wall_area_ft2",
    "ceil_area_ft2",
    "u_win_btu",
    "ach50_numeric",
    "duct_leakage_frac",
    "year_built_numeric",
    "calculated_shd",
    # "in.occupants",
    # "in.heating_setpoint",

]

categorical_features = [
    "in.geometry_building_type_recs",
    "in.geometry_wall_type",
    "in.geometry_attic_type",
    "in.hvac_heating_type_and_fuel",
    "in.hvac_has_ducts",
    "in.duct_location",
    "in.insulation_wall",
    "in.insulation_ceiling",
    "in.insulation_roof",
    "in.vintage",
    "in.state",
    "in.geometry_wall_exterior_finish",
    "in.roof_material",
    "in.windows",
]

numeric_features     = [c for c in numeric_features     if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
features = numeric_features + categorical_features

df = df.dropna(subset=["specific_heat_demand_kwh_m2", CLIMATE_COL]).copy()
print(f"Rows ready: {len(df):,}")
print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")
print(f"Total features: {len(features)}")

if "bldg_id" in df.columns:
    _bldg_ids = df["bldg_id"].unique()
    _train_bldgs, _test_bldgs = train_test_split(
        _bldg_ids, test_size=0.20, random_state=RANDOM_STATE
    )
    train_idx = df[df["bldg_id"].isin(_train_bldgs)].index
    test_idx  = df[df["bldg_id"].isin(_test_bldgs)].index
    print(f"Split by bldg_id: {len(_train_bldgs):,} train buildings, {len(_test_bldgs):,} test buildings")
else:
    # Fallback: stratified row split if bldg_id unavailable
    counts = df[CLIMATE_COL].value_counts()
    stratify = df[CLIMATE_COL] if counts.min() >= 2 else None
    train_idx, test_idx = train_test_split(
        df.index, test_size=0.20, random_state=RANDOM_STATE, stratify=stratify
    )
    print("WARNING: bldg_id not found — falling back to row-level split (leakage risk)")

df_train = df.loc[train_idx].copy()
df_test  = df.loc[test_idx].copy()

# Clip outliers in TRAINING set only — test set is never filtered.
# Filtering test rows removes the hardest cases and inflates metrics.
q1 = df_train["specific_heat_demand_kwh_m2"].quantile(0.005)
q2 = df_train["specific_heat_demand_kwh_m2"].quantile(0.995)
df_train = df_train[
    (df_train["specific_heat_demand_kwh_m2"] >= q1) &
    (df_train["specific_heat_demand_kwh_m2"] <= q2)
].copy()
print(f"Train rows after outlier clip: {len(df_train):,} | Test rows (unfiltered): {len(df_test):,}")

# ---------------- Optuna tuning ----------------
def tune_model(model_name, param_space_fn, X_train, y_train, groups_train, num_feats, cat_feats, n_trials_cat=N_TRIALS_CAT_GLOBAL):
    cv = get_cv(groups_train, CV_FOLDS, RANDOM_STATE)
    groups_for_cv = randomize_groups(groups_train, RANDOM_STATE) if groups_train is not None else None

    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction="minimize", pruner=pruner)

    def objective(trial):
        params = param_space_fn(trial)
        rmses = []
        best_iters = []

        for fold_i, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train, groups_for_cv)):
            X_tr = X_train.iloc[tr_idx]
            y_tr = y_train[tr_idx]
            X_va = X_train.iloc[va_idx]
            y_va = y_train[va_idx]

            y_tr_fit = np.log1p(y_tr) if USE_LOG_TARGET else y_tr
            y_va_fit = np.log1p(y_va) if USE_LOG_TARGET else y_va

            if model_name in ("xgb", "lgb"):
                prep = clone(build_preprocess_ohe(num_feats, cat_feats))
                X_tr_p = prep.fit_transform(X_tr)
                X_va_p = prep.transform(X_va)

                if model_name == "xgb":
                    # early_stopping_rounds moved to constructor in XGBoost >= 2.0
                    model = xgb.XGBRegressor(
                        **params, n_estimators=MAX_ESTIMATORS, tree_method="hist",
                        n_jobs=-1, random_state=RANDOM_STATE,
                        early_stopping_rounds=EARLY_STOPPING_ROUNDS
                    )
                    model.fit(X_tr_p, y_tr_fit, eval_set=[(X_va_p, y_va_fit)], verbose=False)
                else:
                    model = LGBMRegressor(
                        **params, n_estimators=MAX_ESTIMATORS,
                        n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1
                    )
                    model.fit(X_tr_p, y_tr_fit, eval_set=[(X_va_p, y_va_fit)],
                              callbacks=[
                                  __import__("lightgbm").early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                                  __import__("lightgbm").log_evaluation(period=-1)
                              ])

                best_iter = getattr(model, "best_iteration_", None) or MAX_ESTIMATORS
                best_iters.append(best_iter)
                pred = model.predict(X_va_p)

            else:
                X_tr_p, num_imp, cat_imp, cat_idx = fit_transform_catboost(X_tr, num_feats, cat_feats)
                X_va_p = transform_catboost(X_va, num_feats, cat_feats, num_imp, cat_imp)

                model = CatBoostRegressor(
                    **params, iterations=MAX_ESTIMATORS,
                    verbose=0, random_seed=RANDOM_STATE
                )
                model.fit(X_tr_p, y_tr_fit, eval_set=(X_va_p, y_va_fit),
                          use_best_model=True, cat_features=cat_idx,
                          early_stopping_rounds=EARLY_STOPPING_ROUNDS)

                best_iter = getattr(model, "get_best_iteration", lambda: MAX_ESTIMATORS)()
                best_iters.append(best_iter)
                pred = model.predict(X_va_p)

            if USE_LOG_TARGET:
                pred = np.expm1(pred)
                pred = np.clip(pred, 0, None)

            rmses.append(rmse(y_va, pred))

            trial.report(float(np.mean(rmses)), step=fold_i)
            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("best_iters", best_iters)
        return float(np.mean(rmses))

    study.optimize(objective, n_trials={
        "xgb": N_TRIALS_XGB, "cat": n_trials_cat, "lgb": N_TRIALS_LGB
    }[model_name], show_progress_bar=False)

    best_iters = study.best_trial.user_attrs.get("best_iters", [MAX_ESTIMATORS])
    best_iter_avg = int(np.mean(best_iters))

    return study.best_params, study.best_value, best_iter_avg

# ---------------- Search spaces ----------------
def xgb_space(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 40.0),
        "gamma": trial.suggest_float("gamma", 0.0, 15.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 30.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 60.0, log=True),
    }

def lgb_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_samples": trial.suggest_int("min_child_samples", 40, 300),
        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 30.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 60.0, log=True),
    }

def make_cat_space(max_depth):
    def cat_space(trial):
        return {
            "depth": trial.suggest_int("depth", 4, max_depth),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 5.0, 60.0),
            "random_strength": trial.suggest_float("random_strength", 1.0, 12.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 6.0),
            "loss_function": "RMSE",
        }
    return cat_space


# ---------------- Final model fitting ----------------
def fit_final_model(model_name, params, best_iter, X_tr, y_tr, num_feats, cat_feats):
    y_tr_fit = np.log1p(y_tr) if USE_LOG_TARGET else y_tr

    if model_name in ("xgb", "lgb"):
        prep = clone(build_preprocess_ohe(num_feats, cat_feats))
        X_tr_p = prep.fit_transform(X_tr)

        if model_name == "xgb":
            model = xgb.XGBRegressor(
                **params, n_estimators=best_iter, tree_method="hist",
                n_jobs=-1, random_state=RANDOM_STATE
            )
        else:
            model = LGBMRegressor(
                **params, n_estimators=best_iter,
                n_jobs=-1, random_state=RANDOM_STATE
            )

        model.fit(X_tr_p, y_tr_fit)
        pipe = Pipeline([("preprocess", prep), ("model", model)])
        return pipe

    else:
        X_tr_p, num_imp, cat_imp, cat_idx = fit_transform_catboost(X_tr, num_feats, cat_feats)
        model = CatBoostRegressor(
            **params, iterations=best_iter,
            verbose=0, random_seed=RANDOM_STATE
        )
        model.fit(X_tr_p, y_tr_fit, cat_features=cat_idx)

        return {
            "num_feats": num_feats,
            "cat_feats": cat_feats,
            "num_imp": num_imp,
            "cat_imp": cat_imp,
            "model": model,
        }

def predict_pipe(pipe, X):
    if isinstance(pipe, dict):
        X_p = transform_catboost(X, pipe["num_feats"], pipe["cat_feats"],
                                 pipe["num_imp"], pipe["cat_imp"])
        pred = pipe["model"].predict(X_p)
    else:
        pred = pipe.predict(X)
    if USE_LOG_TARGET:
        pred = np.expm1(pred)
        pred = np.clip(pred, 0, None)
    return pred

# ---------------- Train + evaluate ----------------
def train_and_eval_all(df_tr, df_te, label, results, is_zone=False):
    num_feats = [c for c in numeric_features if c in df_tr.columns]
    cat_feats = [c for c in categorical_features if c in df_tr.columns]

    X_tr = df_tr[num_feats + cat_feats]
    y_tr = df_tr["specific_heat_demand_kwh_m2"].to_numpy(dtype=np.float64)
    X_te = df_te[num_feats + cat_feats]
    y_te = df_te["specific_heat_demand_kwh_m2"].to_numpy(dtype=np.float64)

    groups_tr = df_tr["bldg_id"].to_numpy() if "bldg_id" in df_tr.columns else None

    # Dynamic CatBoost settings based on dataset size
    n_samples = len(X_tr)
    cat_max_depth = 6 if (is_zone or n_samples < 100_000) else 7
    n_trials_cat = N_TRIALS_CAT_ZONE if is_zone else N_TRIALS_CAT_GLOBAL
    print(f"CatBoost: max_depth={cat_max_depth}, n_trials={n_trials_cat} (n={n_samples:,})")


    print(f"\n=== {label} ===")
    baseline_pred = np.full_like(y_te, np.median(y_tr), dtype=np.float64)
    baseline_rmse = rmse(y_te, baseline_pred)
    print(f"Baseline RMSE (median): {baseline_rmse:.3f}")

    xgb_best, xgb_cv, xgb_iter = tune_model("xgb", xgb_space, X_tr, y_tr, groups_tr, num_feats, cat_feats)
    cat_best, cat_cv, cat_iter = tune_model("cat", make_cat_space(cat_max_depth), X_tr, y_tr, groups_tr, num_feats, cat_feats, n_trials_cat=n_trials_cat)
    lgb_best, lgb_cv, lgb_iter = tune_model("lgb", lgb_space, X_tr, y_tr, groups_tr, num_feats, cat_feats)

    print(f"Best CV RMSE -> XGB:{xgb_cv:.3f} | CAT:{cat_cv:.3f} | LGB:{lgb_cv:.3f}")

    cv_scores = {"XGBoost": xgb_cv, "CatBoost": cat_cv, "LightGBM": lgb_cv}
    best_by_cv = min(cv_scores, key=cv_scores.get)

    models = {
        "XGBoost": ("xgb", xgb_best, xgb_iter),
        "CatBoost": ("cat", cat_best, cat_iter),
        "LightGBM": ("lgb", lgb_best, lgb_iter),
    }

    for name, (mtype, params, best_iter) in models.items():
        pipe = fit_final_model(mtype, params, best_iter, X_tr, y_tr, num_feats, cat_feats)

        preds = predict_pipe(pipe, X_te)
        preds_tr = predict_pipe(pipe, X_tr)

        mae = float(mean_absolute_error(y_te, preds))
        rmse_te = rmse(y_te, preds)
        r2 = float(r2_score(y_te, preds))
        rmse_tr = rmse(y_tr, preds_tr)

        print(f"{name:9s} | Train RMSE:{rmse_tr:8.3f} | Test RMSE:{rmse_te:8.3f} | "
              f"MAE:{mae:8.3f} | R²:{r2:7.4f}")

        joblib.dump(pipe, MODEL_SAVE_DIR / f"{name.lower()}_{label.replace(' ', '_')}.joblib")

        plot_path = PLOTS_DIR / f"actual_vs_pred_{label.replace(' ', '_')}_{name}.png"
        make_actual_vs_pred_plot(y_te, preds, f"{label} - {name}", plot_path)

        results.append({
            "run_id": run_id,
            "label": label,
            "model": name,
            "cv_rmse": cv_scores[name],
            "train_rmse": rmse_tr,
            "test_rmse": rmse_te,
            "mae": mae,
            "r2": r2,
            "baseline_rmse": baseline_rmse,
            "best_by_cv": (name == best_by_cv),
            "best_iter": best_iter,
        })

=== Loading parquet ===
Raw shape: (549718, 290)
Dropped 0 rows with NaN in calculated features (549,713 remaining)
Rows ready: 549,713
Numeric features: ['floor_area_m2', 'hdd65', 'HDD65_x_UA_per_area', 'UA_total', 'UA_per_area', 'r_wall', 'r_ceiling', 'r_roof', 'r_floor', 'r_foundation_wall', 'r_slab', 'r_rim_joist', 'wall_area_ft2', 'win_area_ft2', 'net_wall_area_ft2', 'ceil_area_ft2', 'u_win_btu', 'ach50_numeric', 'duct_leakage_frac', 'year_built_numeric', 'calculated_shd']
Categorical features: ['in.geometry_building_type_recs', 'in.geometry_wall_type', 'in.geometry_attic_type', 'in.hvac_heating_type_and_fuel', 'in.hvac_has_ducts', 'in.duct_location', 'in.insulation_wall', 'in.insulation_ceiling', 'in.insulation_roof', 'in.vintage', 'in.state', 'in.geometry_wall_exterior_finish', 'in.roof_material', 'in.windows']
Total features: 35
Split by bldg_id: 439,770 train buildings, 109,943 test buildings
Train rows after outlier clip: 437,571 | Test rows (unfiltered): 109,943


In [8]:
_mask = df["calculated_shd"].notna()
print(f"calculated_shd : {_mask.sum():,} valid rows")
print(f"  median calc  : {df.loc[_mask, 'calculated_shd'].median():.1f} kWh/m²/yr")
print(f"  median actual: {df.loc[_mask, 'specific_heat_demand_kwh_m2'].median():.1f} kWh/m²/yr")
print(f"  Pearson r    : {df.loc[_mask, 'calculated_shd'].corr(df.loc[_mask, 'specific_heat_demand_kwh_m2']):.3f}")

calculated_shd : 549,713 valid rows
  median calc  : 68.0 kWh/m²/yr
  median actual: 59.6 kWh/m²/yr
  Pearson r    : 0.772


Plot the scatter of actual vs calculated


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract variables directly from the global dataframe
y_actual = df["specific_heat_demand_kwh_m2"]
y_calc = df["calculated_shd"]

plt.figure(figsize=(8, 8))

sns.scatterplot(x=y_actual, y=y_calc, alpha=0.3, s=15, edgecolor=None, color="blue")

# Calculate the bounds for the 1:1 line
# min_val = min(y_actual.min(), y_calc.min())
# max_val = max(y_actual.max(), y_calc.max())

# # Perfect match reference line
# plt.plot(
#     [min_val, max_val], [min_val, max_val], 
#     color='red', linestyle='--', linewidth=2, label='Perfect Match (y=x)'
# )

# Max 800
plt.xlim(0, 800)
plt.ylim(0, 800)
plt.plot(
    [0, 800], [0, 800], 
    color='red', linestyle='--', linewidth=2, label='Perfect Match (y=x)'
)
plt.xlabel("Actual Specific Heating Demand (kWh/m²/yr)", fontsize=12)
plt.ylabel("Calculated Physics SHD (kWh/m²/yr)", fontsize=12)
plt.title("Actual vs. Calculated Physics Estimate", fontsize=14)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()

plt.show()

In [42]:
# ================================================================
# PRE-FLIGHT TEST CELL
# Run this AFTER the setup cell above, BEFORE the training cell below.
# All checks must print OK. Fix any FAIL before proceeding.
# ================================================================
import psutil

_preflight_results = []

def _check(name, fn):
    try:
        msg = fn()
        print(f"  OK   {name}" + (f" — {msg}" if msg else ""))
        _preflight_results.append((name, True))
    except Exception as e:
        print(f"  FAIL {name} — {e}")
        _preflight_results.append((name, False))

print("=" * 60)
print("LIBRARY VERSIONS")
print("=" * 60)
_check("XGBoost",     lambda: __import__("xgboost").__version__)
_check("LightGBM",    lambda: __import__("lightgbm").__version__)
_check("CatBoost",    lambda: __import__("catboost").__version__)
_check("Optuna",      lambda: __import__("optuna").__version__)
_check("Scikit-learn",lambda: __import__("sklearn").__version__)
_check("Joblib",      lambda: __import__("joblib").__version__)

print()
print("=" * 60)
print("MEMORY")
print("=" * 60)
def _chk_ram():
    mem = psutil.virtual_memory()
    free_gb = mem.available / 1e9
    total_gb = mem.total / 1e9
    if free_gb < 4:
        raise RuntimeError(f"Only {free_gb:.1f} GB free — close other apps before training")
    return f"{free_gb:.1f} GB free of {total_gb:.1f} GB total"
_check("Available RAM", _chk_ram)

print()
print("=" * 60)
print("FILES & PATHS")
print("=" * 60)
def _chk_data():
    p = Path(DATA_PATH)
    if not p.exists(): raise FileNotFoundError(f"{DATA_PATH}")
    return f"{p.stat().st_size / 1e9:.2f} GB"
def _chk_output():
    MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    t = MODEL_SAVE_DIR / ".write_test"
    t.write_text("ok"); t.unlink()
    return str(MODEL_SAVE_DIR)
_check("Data file exists", _chk_data)
_check("Output dir writable", _chk_output)

print()
print("=" * 60)
print("PARQUET SCHEMA")
print("=" * 60)
def _chk_parquet():
    _df = pd.read_parquet(DATA_PATH).convert_dtypes()
    r, c = _df.shape
    heating = ["calc.enduse_group.site_energy.heating.energy_consumption..kwh"]
    if not heating: raise ValueError("No heating energy column found")
    if "in.sqft" not in _df.columns: raise ValueError("in.sqft missing")
    if CLIMATE_COL not in _df.columns: raise ValueError(f"{CLIMATE_COL} missing")
    zones = _df[CLIMATE_COL].nunique()
    return f"{r:,} rows x {c} cols | {zones} climate zones | heating: {heating[0]}"
_check("Parquet loads + schema", _chk_parquet)

print()
print("=" * 60)
print("MODEL SMOKE TESTS  (~15 seconds)")
print("=" * 60)
def _chk_xgb():
    import xgboost as xgb
    rng = np.random.default_rng(0); X = rng.random((300,10)); y = rng.random(300)
    m = xgb.XGBRegressor(n_estimators=10, tree_method="hist", verbosity=0,
                         early_stopping_rounds=5, random_state=0)
    m.fit(X[:240], y[:240], eval_set=[(X[240:], y[240:])], verbose=False)
    return f"predict shape {m.predict(X[240:]).shape}"

def _chk_lgb():
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    rng = np.random.default_rng(1); X = rng.random((300,10)); y = rng.random(300)
    m = LGBMRegressor(n_estimators=20, verbosity=-1, random_state=0)
    m.fit(X[:240], y[:240], eval_set=[(X[240:], y[240:])],
          callbacks=[early_stopping(5, verbose=False), log_evaluation(period=-1)])
    return f"best_iteration_={m.best_iteration_}"

def _chk_cat():
    from catboost import CatBoostRegressor
    rng = np.random.default_rng(2); X = rng.random((300,10)); y = rng.random(300)
    m = CatBoostRegressor(iterations=10, verbose=0, random_seed=0)
    m.fit(X[:240], y[:240], eval_set=(X[240:], y[240:]), use_best_model=True)
    return f"predict shape {m.predict(X[240:]).shape}"

def _chk_optuna_lgb():
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    rng = np.random.default_rng(3); X = rng.random((300,8)); y = rng.random(300)
    def _obj(trial):
        m = LGBMRegressor(n_estimators=30,
                          learning_rate=trial.suggest_float("lr", 0.01, 0.1),
                          verbosity=-1)
        m.fit(X[:240], y[:240], eval_set=[(X[240:], y[240:])],
              callbacks=[early_stopping(5, verbose=False), log_evaluation(period=-1)])
        return float(np.sqrt(np.mean((m.predict(X[240:]) - y[240:]) ** 2)))
    s = optuna.create_study(direction="minimize")
    s.optimize(_obj, n_trials=3, show_progress_bar=False)
    return f"3 trials, best={s.best_value:.4f}"

def _chk_cat_space_cap():
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
    for cap in [6, 8]:
        fn = make_cat_space(cap)
        s = optuna.create_study(); trial = s.ask()
        p = fn(trial)
        assert p["depth"] <= cap, f"depth {p['depth']} exceeds cap {cap}"
    return "depth cap enforced for both global (8) and zone (6)"

def _chk_joblib():
    import joblib
    from sklearn.linear_model import Ridge
    path = MODEL_SAVE_DIR / "_preflight.joblib"
    joblib.dump(Ridge().fit([[1],[2],[3]], [1,2,3]), path)
    joblib.load(path); path.unlink()
    return "save + load OK"

_check("XGBoost fit + early stopping", _chk_xgb)
_check("LightGBM fit + callbacks (bug fix verified)", _chk_lgb)
_check("CatBoost fit + eval_set", _chk_cat)
_check("Optuna + LightGBM end-to-end", _chk_optuna_lgb)
_check("make_cat_space depth cap", _chk_cat_space_cap)
_check("Joblib save/load", _chk_joblib)

print()
print("=" * 60)
_failed = [n for n, ok in _preflight_results if not ok]
if _failed:
    print("PREFLIGHT FAILED — fix before running training:")
    for _f in _failed:
        print(f"  ✗ {_f}")
else:
    print("ALL CHECKS PASSED — safe to run the training cell")
print("=" * 60)

# Restore Optuna logging for the training cell
import optuna as _optuna
_optuna.logging.set_verbosity(_optuna.logging.INFO)
print("Optuna logging restored to INFO for training cell.")


LIBRARY VERSIONS
  OK   XGBoost — 1.7.6
  OK   LightGBM — 4.6.0
  OK   CatBoost — 1.2.10
  OK   Optuna — 4.8.0
  OK   Scikit-learn — 1.7.2
  OK   Joblib — 1.5.2

MEMORY
  OK   Available RAM — 12.8 GB free of 34.1 GB total

FILES & PATHS
  OK   Data file exists — 0.17 GB
  OK   Output dir writable — A:\College\Thesis\Submission\models\resstock_5.8

PARQUET SCHEMA
  OK   Parquet loads + schema — 549,718 rows x 289 cols | 7 climate zones | heating: calc.enduse_group.site_energy.heating.energy_consumption..kwh

MODEL SMOKE TESTS  (~15 seconds)
  OK   XGBoost fit + early stopping — predict shape (60,)
  OK   LightGBM fit + callbacks (bug fix verified) — best_iteration_=2
  OK   CatBoost fit + eval_set — predict shape (60,)
  OK   Optuna + LightGBM end-to-end — 3 trials, best=0.2706
  OK   make_cat_space depth cap — depth cap enforced for both global (8) and zone (6)
  OK   Joblib save/load — save + load OK

ALL CHECKS PASSED — safe to run the training cell
Optuna logging restored to INFO fo

In [ ]:
# ================================================================
# TRAINING: Global Model
# ================================================================
results = []

train_and_eval_all(df_train, df_test, "global", results, is_zone=False)

results_df = pd.DataFrame(results)
results_path = MODEL_SAVE_DIR / f"model_results_global_{run_id}.csv"
results_df.to_csv(results_path, index=False)
print(f"\n=== GLOBAL TRAINING COMPLETE ===")
print(f"Results CSV: {results_path}")


In [13]:
# ================================================================
# TRAINING: Zone — Marine
# ================================================================
_zone_results = []
_zone = "Marine"
_zone_label = "Marine"

dz = df[df[CLIMATE_COL].astype("string") == _zone].copy()
dz_train = dz.loc[train_idx.intersection(dz.index)]
dz_test  = dz.loc[test_idx.intersection(dz.index)]

if len(dz_train) < MIN_SAMPLES_PER_ZONE or len(dz_test) == 0:
    print(f"Skipping zone {_zone}: insufficient samples "
          f"(train={len(dz_train)}, test={len(dz_test)})")
else:
    train_and_eval_all(dz_train, dz_test, f"zone_{_zone_label}", _zone_results, is_zone=True)
    _zone_df = pd.DataFrame(_zone_results)
    _zone_path = MODEL_SAVE_DIR / f"model_results_zone_{_zone_label}_{run_id}.csv"
    _zone_df.to_csv(_zone_path, index=False)
    print(f"\n=== ZONE {_zone} TRAINING COMPLETE ===")
    print(f"Results CSV: {_zone_path}")


[I 2026-05-14 20:19:51,606] A new study created in memory with name: no-name-0648b841-ea97-4ad7-8942-efdc03fdb606


CatBoost: max_depth=6, n_trials=30 (n=23,087)

=== zone_Marine ===
Baseline RMSE (median): 51.955


[I 2026-05-14 20:20:03,489] Trial 0 finished with value: 28.247661194087108 and parameters: {'max_depth': 6, 'learning_rate': 0.05669989107282507, 'subsample': 0.8399870023380434, 'colsample_bytree': 0.7697330433946838, 'min_child_weight': 9.56469602136546, 'gamma': 2.8174697274742364, 'reg_alpha': 0.004563309871804098, 'reg_lambda': 0.02735425706799479}. Best is trial 0 with value: 28.247661194087108.
[I 2026-05-14 20:20:23,217] Trial 1 finished with value: 27.81150282292785 and parameters: {'max_depth': 4, 'learning_rate': 0.0518343930160649, 'subsample': 0.7546656732812518, 'colsample_bytree': 0.6335985024105162, 'min_child_weight': 37.56846590561121, 'gamma': 0.5938353433480015, 'reg_alpha': 8.464972839628295, 'reg_lambda': 3.263477090391488}. Best is trial 1 with value: 27.81150282292785.
[I 2026-05-14 20:20:46,653] Trial 2 finished with value: 29.204633949642563 and parameters: {'max_depth': 7, 'learning_rate': 0.028740322676317168, 'subsample': 0.7866194192907726, 'colsample_byt

TypeError: transform_catboost() takes 4 positional arguments but 5 were given

In [ ]:
# ================================================================
# TRAINING: Zone — Cold & Very Cold
# ================================================================
_zone_results = []
_zone = "Cold & Very Cold"
_zone_label = "Cold_and_Very_Cold"

# Filter dataframe for both zones using isin()
dz = df[df[CLIMATE_COL].astype("string").isin(["Cold", "Very Cold"])].copy()

# Add a binary feature indicating if the original zone was 'Very Cold'
dz["is_very_cold"] = (dz[CLIMATE_COL].astype("string") == "Very Cold").astype(int)

# Temporarily register the new feature so train_and_eval_all picks it up
_added_feature = False
if "is_very_cold" not in numeric_features:
    numeric_features.append("is_very_cold")
    _added_feature = True

dz_train = dz.loc[train_idx.intersection(dz.index)]
dz_test  = dz.loc[test_idx.intersection(dz.index)]

if len(dz_train) < MIN_SAMPLES_PER_ZONE or len(dz_test) == 0:
    print(f"Skipping zone {_zone}: insufficient samples "
          f"(train={len(dz_train)}, test={len(dz_test)})")
else:
    train_and_eval_all(dz_train, dz_test, f"zone_{_zone_label}", _zone_results, is_zone=True)
    
    _zone_df = pd.DataFrame(_zone_results)
    _zone_path = MODEL_SAVE_DIR / f"model_results_zone_{_zone_label}_{run_id}.csv"
    _zone_df.to_csv(_zone_path, index=False)
    
    print(f"\n=== ZONE {_zone} TRAINING COMPLETE ===")
    print(f"Results CSV: {_zone_path}")

# Remove the feature from the global list to prevent errors in subsequent zone training cells
if _added_feature:
    numeric_features.remove("is_very_cold")

In [ ]:
# ================================================================
# TRAINING: Zone — Mixed-Humid
# ================================================================
_zone_results = []
_zone = "Mixed-Humid"
_zone_label = "Mixed-Humid"

dz = df[df[CLIMATE_COL].astype("string") == _zone].copy()
dz_train = dz.loc[train_idx.intersection(dz.index)]
dz_test  = dz.loc[test_idx.intersection(dz.index)]

if len(dz_train) < MIN_SAMPLES_PER_ZONE or len(dz_test) == 0:
    print(f"Skipping zone {_zone}: insufficient samples "
          f"(train={len(dz_train)}, test={len(dz_test)})")
else:
    train_and_eval_all(dz_train, dz_test, f"zone_{_zone_label}", _zone_results, is_zone=True)
    _zone_df = pd.DataFrame(_zone_results)
    _zone_path = MODEL_SAVE_DIR / f"model_results_zone_{_zone_label}_{run_id}.csv"
    _zone_df.to_csv(_zone_path, index=False)
    print(f"\n=== ZONE {_zone} TRAINING COMPLETE ===")
    print(f"Results CSV: {_zone_path}")


In [ ]:
# ================================================================
# TRAINING: Zone — Mixed-Dry
# ================================================================
_zone_results = []
_zone = "Mixed-Dry"
_zone_label = "Mixed-Dry"

dz = df[df[CLIMATE_COL].astype("string") == _zone].copy()
dz_train = dz.loc[train_idx.intersection(dz.index)]
dz_test  = dz.loc[test_idx.intersection(dz.index)]

if len(dz_train) < MIN_SAMPLES_PER_ZONE or len(dz_test) == 0:
    print(f"Skipping zone {_zone}: insufficient samples "
          f"(train={len(dz_train)}, test={len(dz_test)})")
else:
    train_and_eval_all(dz_train, dz_test, f"zone_{_zone_label}", _zone_results, is_zone=True)
    _zone_df = pd.DataFrame(_zone_results)
    _zone_path = MODEL_SAVE_DIR / f"model_results_zone_{_zone_label}_{run_id}.csv"
    _zone_df.to_csv(_zone_path, index=False)
    print(f"\n=== ZONE {_zone} TRAINING COMPLETE ===")
    print(f"Results CSV: {_zone_path}")


In [ ]:
# ================================================================
# TRAINING: Zone — Hot-Humid
# ================================================================
_zone_results = []
_zone = "Hot-Humid"
_zone_label = "Hot-Humid"

dz = df[df[CLIMATE_COL].astype("string") == _zone].copy()
dz_train = dz.loc[train_idx.intersection(dz.index)]
dz_test  = dz.loc[test_idx.intersection(dz.index)]

if len(dz_train) < MIN_SAMPLES_PER_ZONE or len(dz_test) == 0:
    print(f"Skipping zone {_zone}: insufficient samples "
          f"(train={len(dz_train)}, test={len(dz_test)})")
else:
    train_and_eval_all(dz_train, dz_test, f"zone_{_zone_label}", _zone_results, is_zone=True)
    _zone_df = pd.DataFrame(_zone_results)
    _zone_path = MODEL_SAVE_DIR / f"model_results_zone_{_zone_label}_{run_id}.csv"
    _zone_df.to_csv(_zone_path, index=False)
    print(f"\n=== ZONE {_zone} TRAINING COMPLETE ===")
    print(f"Results CSV: {_zone_path}")


In [ ]:
# ================================================================
# TRAINING: Zone — Hot-Dry
# ================================================================
_zone_results = []
_zone = "Hot-Dry"
_zone_label = "Hot-Dry"

dz = df[df[CLIMATE_COL].astype("string") == _zone].copy()
dz_train = dz.loc[train_idx.intersection(dz.index)]
dz_test  = dz.loc[test_idx.intersection(dz.index)]

if len(dz_train) < MIN_SAMPLES_PER_ZONE or len(dz_test) == 0:
    print(f"Skipping zone {_zone}: insufficient samples "
          f"(train={len(dz_train)}, test={len(dz_test)})")
else:
    train_and_eval_all(dz_train, dz_test, f"zone_{_zone_label}", _zone_results, is_zone=True)
    _zone_df = pd.DataFrame(_zone_results)
    _zone_path = MODEL_SAVE_DIR / f"model_results_zone_{_zone_label}_{run_id}.csv"
    _zone_df.to_csv(_zone_path, index=False)
    print(f"\n=== ZONE {_zone} TRAINING COMPLETE ===")
    print(f"Results CSV: {_zone_path}")


In [ ]:
# ================================================================
# SHAP ANALYSIS — Comprehensive Interpretability Suite
# ================================================================
# Analyses every saved model (global + all zones) across all three
# model types: XGBoost, LightGBM, CatBoost.
#
# Plots generated per model:
#   1. Bar plot          — mean |SHAP| feature importance ranking
#   2. Beeswarm / summary plot — magnitude + direction per feature
#   3. Heatmap           — feature × sample SHAP interaction overview
#   4. Dependence plots  — top-5 features (auto-selects best interaction)
#   5. Waterfall plot    — single-sample local explanation (median pred)
#   6. Force plot (HTML) — interactive local explanation saved as HTML
#
# Cross-zone summary:
#   7. Grouped bar chart — top-10 features ranked by zone for comparison
#   8. CSV export        — raw mean |SHAP| table for all models × features
#
# Usage: run after all training cells have completed.
# All outputs land in  MODEL_SAVE_DIR / "shap_analysis"
# ================================================================

import shap
import joblib
import matplotlib
matplotlib.use("Agg")          # headless — avoids Tk/Qt dependency
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import warnings
import traceback
from pathlib import Path
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", message=".*check_additivity.*")
warnings.filterwarnings("ignore", message=".*LightGBM.*")
warnings.filterwarnings("ignore", category=UserWarning)

# ── Output directory ─────────────────────────────────────────────
SHAP_DIR = MODEL_SAVE_DIR / "SHAP_results"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

# ── Sampling ─────────────────────────────────────────────────────
SHAP_SAMPLE_SIZE   = 2000   # rows used for global SHAP (beeswarm/bar)
SHAP_INTERACT_SIZE = 500    # rows for interaction values (expensive)
RANDOM_STATE_SHAP  = 42


# ════════════════════════════════════════════════════════════════
# Helper: reconstruct test slice for a given zone label
# ════════════════════════════════════════════════════════════════

def transform_catboost(X, num_feats, cat_feats, num_imp, cat_imp):
    X_num = pd.DataFrame(num_imp.transform(X[num_feats]), columns=num_feats, index=X.index)
    X_cat = pd.DataFrame(cat_imp.transform(X[cat_feats]), columns=cat_feats, index=X.index).astype("string")
    return pd.concat([X_num, X_cat], axis=1)

def _get_zone_test_df(zone_label: str) -> pd.DataFrame:
    """
    Returns the test-set slice for a zone.  Handles:
      • 'global'           → full test set
      • 'Cold_and_Very_Cold' → union of Cold + Very Cold + is_very_cold flag
      • everything else    → single-zone match (underscores → hyphens/spaces)
    """
    if zone_label == "global":
        return df.loc[test_idx].copy()

    if zone_label == "Cold_and_Very_Cold":
        dz = df[df[CLIMATE_COL].astype("string").isin(["Cold", "Very Cold"])].copy()
        dz["is_very_cold"] = (dz[CLIMATE_COL].astype("string") == "Very Cold").astype(int)
        return dz.loc[test_idx.intersection(dz.index)].copy()

    # Try the label as-is first (e.g. "Hot-Humid"), then with underscores
    # replaced by hyphens or spaces to match how the parquet stores the value.
    candidates = [
        zone_label,
        zone_label.replace("_", "-"),
        zone_label.replace("_", " "),
    ]
    for candidate in candidates:
        mask = df[CLIMATE_COL].astype("string") == candidate
        if mask.any():
            dz = df[mask].copy()
            return dz.loc[test_idx.intersection(dz.index)].copy()

    return pd.DataFrame()   # not found


# ════════════════════════════════════════════════════════════════
# Helper: extract (X_df, explainer, shap_values) from a saved pipe
# ════════════════════════════════════════════════════════════════
def _prepare_shap(pipe_or_dict, dz_test: pd.DataFrame, sample_size: int):
    import re
    is_catboost = isinstance(pipe_or_dict, dict)

    if is_catboost:
        # Extract features and BOTH imputers
        num_f = pipe_or_dict["num_feats"]
        cat_f = pipe_or_dict["cat_feats"]
        all_feats = num_f + cat_f
    else:
        pipe = pipe_or_dict
        preprocessor = pipe.named_steps["preprocess"]
        if hasattr(preprocessor, "feature_names_in_"):
            all_feats = list(preprocessor.feature_names_in_)
        else:
            num_f = preprocessor.transformers_[1][2]
            cat_f = preprocessor.transformers_[0][2]
            all_feats = list(num_f) + list(cat_f)

    all_feats = [c for c in all_feats if c in dz_test.columns]
    n = min(sample_size, len(dz_test))
    X_sample = dz_test[all_feats].sample(n, random_state=RANDOM_STATE_SHAP)

    if is_catboost:
        num_f = pipe_or_dict["num_feats"]
        cat_f = pipe_or_dict["cat_feats"]
        X_transformed = transform_catboost(
            X_sample,
            num_f,
            cat_f,
            pipe_or_dict.get("num_imp"),
            pipe_or_dict.get("cat_imp")
        )
        model = pipe_or_dict["model"]
        
        feat_names = [re.sub(r'[\[\]<]', '_', str(n)) for n in X_transformed.columns]
        X_out = pd.DataFrame(X_transformed.values, columns=feat_names, index=X_sample.index)
        
        explainer = shap.TreeExplainer(model)
        shap_obj = explainer(X_out, check_additivity=False)
        model = pipe_or_dict["model"]
        feat_names = [re.sub(r'[\[\]<]', '_', str(n)) for n in X_transformed.columns]
        X_out = pd.DataFrame(X_transformed.values, columns=feat_names, index=X_sample.index)
        explainer = shap.TreeExplainer(model)
        shap_obj = explainer(X_out, check_additivity=False)

    else:
        preprocessor = pipe.named_steps["preprocess"]
        model = pipe.named_steps["model"]
        X_transformed = preprocessor.transform(X_sample)

        # Convert sparse output to dense for SHAP
        if hasattr(X_transformed, "toarray"):
            X_transformed = X_transformed.toarray()

        try:
            raw_names = preprocessor.get_feature_names_out()
            # Clean feature names (remove prefixes like 'num__' or 'cat__')
            feat_names = [n.split("__", 1)[-1] if "__" in n else n for n in raw_names]
            # XGBoost specific fix for invalid characters
            feat_names = [re.sub(r'[\[\]<]', '_', str(n)) for n in feat_names]
        except AttributeError:
            feat_names = [f"f{i}" for i in range(X_transformed.shape[1])]

        X_out = pd.DataFrame(X_transformed, columns=feat_names, index=X_sample.index)
        explainer = shap.TreeExplainer(model)
        shap_obj = explainer(X_out, check_additivity=False)

    return X_out, explainer, shap_obj, feat_names
# ════════════════════════════════════════════════════════════════
# Helper: mean absolute SHAP ranking (collapses OHE dummies back
# to the original categorical column name)
# ════════════════════════════════════════════════════════════════
def _mean_abs_shap(shap_obj, feat_names):
    """
    Returns a pd.Series: original_feature → mean |SHAP|, sorted desc.
    OHE dummies (e.g. 'in.hvac_category_Heat Pump') are collapsed back
    to their parent ('in.hvac_category') by summing their contributions.
    """
    vals = np.abs(shap_obj.values)    # (n_samples, n_features)
    mean_abs = pd.Series(vals.mean(axis=0), index=feat_names)

    # Collapse OHE dummies: 'feature_value' → 'feature'
    collapsed = {}
    for fname, v in mean_abs.items():
        # Heuristic: if the column name ends with '_<something>' and
        # the prefix matches a known base feature, merge.
        base = fname
        for known in numeric_features + categorical_features:
            if fname.startswith(known + "_") or fname == known:
                base = known
                break
        collapsed[base] = collapsed.get(base, 0.0) + v

    return pd.Series(collapsed).sort_values(ascending=False)


# ════════════════════════════════════════════════════════════════
# Plotting functions
# ════════════════════════════════════════════════════════════════
def _save_bar_plot(shap_obj, feat_names, title, out_path, top_n=20):
    shap.plots.bar(shap_obj, max_display=top_n, show=False)
    fig = plt.gcf()
    fig.set_size_inches(10, 7)
    fig.axes[0].set_title(title, fontsize=13, pad=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def _save_beeswarm_plot(shap_obj, title, out_path, top_n=20):
    shap.plots.beeswarm(shap_obj, max_display=top_n, show=False)
    fig = plt.gcf()
    fig.set_size_inches(11, 8)
    fig.axes[0].set_title(title, fontsize=13, pad=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def _save_heatmap_plot(shap_obj, title, out_path, top_n=15):
    shap.plots.heatmap(shap_obj, max_display=top_n, show=False)
    fig = plt.gcf()
    fig.set_size_inches(13, 7)
    fig.axes[0].set_title(title, fontsize=13, pad=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def _save_dependence_plots(shap_obj, X_df, feat_names, prefix, out_dir, top_n=5):
    """
    Dependence plots for top-N features by mean |SHAP|.
    Uses the legacy shap.dependence_plot which correctly accepts ax=.
    Passes integer index for OHE dummies so we always produce top_n plots.
    """
    mean_abs = np.abs(shap_obj.values).mean(axis=0)
    ranked_idxs = np.argsort(-mean_abs)

    plotted = 0
    for fi in ranked_idxs:
        if plotted >= top_n:
            break
        fname = feat_names[fi]
        # Use name if it exists as a column, otherwise pass int index
        col_ref = fname if fname in X_df.columns else fi
        fig, ax = plt.subplots(figsize=(8, 6))
        try:
            shap.dependence_plot(
                col_ref,
                shap_obj.values,
                X_df,
                feature_names=feat_names,
                interaction_index="auto",
                ax=ax,
                show=False,
            )
        except Exception:
            plt.close(fig)
            continue
        display_name = fname if isinstance(fname, str) else feat_names[fi]
        ax.set_title(f"Dependence: {display_name}", fontsize=12)
        fig.tight_layout()
        safe = str(display_name).replace("/", "_").replace(" ", "_").replace(".", "_")
        fig.savefig(out_dir / f"{prefix}_dep_{plotted+1:02d}_{safe}.png",
                    dpi=220, bbox_inches="tight")
        plt.close(fig)
        plotted += 1


def _save_waterfall_plot(shap_obj, X_df, y_true, title, out_path):
    """
    Waterfall for the sample whose prediction is closest to the median.
    """
    base_val = shap_obj.base_values
    if base_val.ndim > 1:
        base_val = base_val[:, 0]
    preds = base_val + shap_obj.values.sum(axis=1)
    if USE_LOG_TARGET:
        preds = np.expm1(preds)
    median_pred = np.median(preds)
    idx = int(np.argmin(np.abs(preds - median_pred)))

    shap.plots.waterfall(shap_obj[idx], show=False)
    fig = plt.gcf()
    fig.set_size_inches(10, 7)
    actual_label = (f"  [actual={float(y_true.iloc[idx]):.1f} kWh/m²/yr]"
                    if hasattr(y_true, "iloc") else "")
    fig.axes[0].set_title(f"{title}{actual_label}", fontsize=12, pad=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def _save_force_html(explainer, shap_obj, X_df, out_path):
    """Save an interactive multi-sample force plot as standalone HTML."""
    try:
        force = shap.force_plot(
            explainer.expected_value,
            shap_obj.values[:200],         # cap at 200 for file size
            X_df.iloc[:200],
            show=False,
        )
        shap.save_html(str(out_path), force)
    except Exception as exc:
        # force_plot may fail with CatBoost / log-target models; skip gracefully
        out_path.write_text(f"<pre>Force plot unavailable: {exc}</pre>")


# ════════════════════════════════════════════════════════════════
# Main analysis loop
# ════════════════════════════════════════════════════════════════
all_importance_rows = []   # collected for cross-zone summary

# Discover all saved model files (global + zone)
model_files = sorted(MODEL_SAVE_DIR.glob("*.joblib"))
model_files = [f for f in model_files if not f.stem.startswith("_")]

if not model_files:
    print(f"No .joblib models found in {MODEL_SAVE_DIR}. Run training cells first.")

for model_path in model_files:
    stem = model_path.stem   # e.g. "xgboost_global" or "catboost_zone_Marine"

    # ── Parse stem ───────────────────────────────────────────────
    # Convention: <model_type>_<scope>   where scope = "global" or "zone_<label>"
    parts = stem.split("_", 1)
    if len(parts) < 2:
        continue

    model_type = parts[0].lower()   # xgboost / catboost / lightgbm
    scope      = parts[1]           # "global" or "zone_Marine" etc.

    if scope.startswith("zone_"):
        zone_label = scope[len("zone_"):]
        scope_tag  = f"zone_{zone_label}"
    else:
        zone_label = "global"
        scope_tag  = "global"

    try:
        # ── Load model ───────────────────────────────────────────
        pipe_or_dict = joblib.load(model_path)

        # ── Get test data ────────────────────────────────────────
        dz_test = _get_zone_test_df(zone_label)
        if len(dz_test) == 0:
            continue

        y_te = dz_test["specific_heat_demand_kwh_m2"]

        # ── Compute SHAP values ──────────────────────────────────
        X_df, explainer, shap_obj, feat_names = _prepare_shap(
            pipe_or_dict, dz_test, SHAP_SAMPLE_SIZE
        )

        # ── Per-model output sub-directory ───────────────────────
        sub_dir = SHAP_DIR / f"{model_type}_{scope_tag}"
        sub_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"{model_type}_{scope_tag}"

        # ── 1. Bar plot ──────────────────────────────────────────
        _save_bar_plot(
            shap_obj, feat_names,
            f"{model_type.upper()} — Feature Importance | {scope_tag}",
            sub_dir / f"{prefix}_bar.png",
        )

        # ── 2. Beeswarm / summary ────────────────────────────────
        _save_beeswarm_plot(
            shap_obj,
            f"{model_type.upper()} — SHAP Summary (beeswarm) | {scope_tag}",
            sub_dir / f"{prefix}_beeswarm.png",
        )

        # ── 3. Heatmap ───────────────────────────────────────────
        n_heat = min(300, len(X_df))
        idx_heat = np.random.RandomState(RANDOM_STATE_SHAP).choice(
            len(X_df), n_heat, replace=False
        )
        shap_heat = shap_obj[idx_heat]
        _save_heatmap_plot(
            shap_heat,
            f"{model_type.upper()} — SHAP Heatmap | {scope_tag}",
            sub_dir / f"{prefix}_heatmap.png",
        )

        # ── 4. Dependence plots (top-5 features) ─────────────────
        _save_dependence_plots(
            shap_obj, X_df, feat_names,
            prefix, sub_dir, top_n=5,
        )

        # ── 5. Waterfall (median-prediction sample) ───────────────
        y_te_aligned = y_te.loc[X_df.index] if X_df.index.isin(y_te.index).all() else y_te
        _save_waterfall_plot(
            shap_obj, X_df, y_te_aligned,
            f"{model_type.upper()} — Local Explanation (median pred) | {scope_tag}",
            sub_dir / f"{prefix}_waterfall.png",
        )

        # ── 6. Force plot (interactive HTML) ─────────────────────
        _save_force_html(
            explainer, shap_obj, X_df,
            sub_dir / f"{prefix}_force.html",
        )

        # ── Collect importance for cross-zone summary ─────────────
        importance_series = _mean_abs_shap(shap_obj, feat_names)
        row = {"model": model_type, "scope": scope_tag}
        row.update(importance_series.to_dict())
        all_importance_rows.append(row)

    except Exception:
        print(f"ERROR processing {stem}:")
        traceback.print_exc()
        continue


# ════════════════════════════════════════════════════════════════
# Cross-zone summary plots
# ════════════════════════════════════════════════════════════════
if all_importance_rows:
    importance_df = pd.DataFrame(all_importance_rows).set_index(["model", "scope"])
    importance_df = importance_df.fillna(0.0)

    # ── 7. CSV export ─────────────────────────────────────────────
    csv_path = SHAP_DIR / "shap_importance_all_models.csv"
    importance_df.to_csv(csv_path)

    # ── 8. Grouped bar chart (top-10 features across zones) ───────
    scope_importance = importance_df.groupby("scope").mean()
    top_features = (
        scope_importance.mean(axis=0)
                        .sort_values(ascending=False)
                        .head(10)
                        .index
    )
    scope_importance_top = scope_importance[top_features]

    n_scopes   = len(scope_importance_top)
    n_features = len(top_features)
    bar_width  = 0.7 / n_scopes
    colors     = cm.tab20(np.linspace(0, 1, n_scopes))

    fig, ax = plt.subplots(figsize=(14, 7))
    x = np.arange(n_features)
    for i, (scope_name, row) in enumerate(scope_importance_top.iterrows()):
        offset = (i - n_scopes / 2 + 0.5) * bar_width
        ax.bar(x + offset, row[top_features].values, bar_width,
               label=scope_name, color=colors[i], alpha=0.88)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f.replace("out.params.", "").replace("in.", "") for f in top_features],
        rotation=35, ha="right", fontsize=9,
    )
    ax.set_ylabel("Mean |SHAP value| (kWh/m²/yr)")
    ax.set_title("Top-10 Feature Importance by Climate Zone (mean across model types)",
                 fontsize=12)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(SHAP_DIR / "shap_cross_zone_comparison.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    # ── Top-10 text summary ───────────────────────────────────────
    global_rank = (
        importance_df.mean(axis=0)
                     .sort_values(ascending=False)
                     .head(10)
    )
    print("TOP-10 FEATURES (averaged across all models & zones)")
    print("─" * 60)
    for rank, (feat, val) in enumerate(global_rank.items(), 1):
        bar = "█" * int(val / global_rank.iloc[0] * 30)
        print(f"  {rank:2d}. {feat:<45s} {val:6.3f}  {bar}")

print(f"\nSHAP results saved to: {SHAP_DIR.absolute()}")

ERROR processing catboost_zone_Marine:


Traceback (most recent call last):
  File "C:\Users\moham\AppData\Local\Temp\ipykernel_4224\2442416448.py", line 339, in <module>
    X_df, explainer, shap_obj, feat_names = _prepare_shap(
                                            ~~~~~~~~~~~~~^
        pipe_or_dict, dz_test, SHAP_SAMPLE_SIZE
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\moham\AppData\Local\Temp\ipykernel_4224\2442416448.py", line 113, in _prepare_shap
    X_transformed = transform_catboost(
        X_sample,
    ...<3 lines>...
        pipe_or_dict.get("cat_imp")
    )
TypeError: transform_catboost() takes 4 positional arguments but 5 were given


TOP-10 FEATURES (averaged across all models & zones)
────────────────────────────────────────────────────────────
   1. in.hvac_heating_type_and_fuel                  1.733  ██████████████████████████████
   2. hdd65                                          0.424  ███████
   3. calculated_shd                                 0.328  █████
   4. in.duct_location                               0.270  ████
   5. in.geometry_building_type_recs                 0.258  ████
   6. in.insulation_roof                             0.184  ███
   7. in.geometry_wall_exterior_finish               0.177  ███
   8. in.insulation_wall                             0.174  ███
   9. in.state                                       0.174  ███
  10. in.windows                                     0.149  ██

SHAP results saved to: A:\College\Thesis\Submission\models\resstock_5.8\SHAP_results
